<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/Lgbm_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 102.5 MB/s eta 0:00:00


In [ ]:
"""
LightGBM v1 — Colab GPU + W&B
================================
Based on competition leader (yunsuxiaozi) approach with key improvements:

  KEPT FROM BASELINE
  ──────────────────
  + Extended digit features: k=-4..3 (8 digits × 11 NUMS = 88 features)
  + Global zero-variance digit drop
  + Frequency-rank categorical encoding (min 5 occurrences)
  + TargetEncoder on all features (CATS + digit features)
  + 5-fold KFold CV
  + Balanced sample weights

  IMPROVEMENTS OVER BASELINE
  ───────────────────────────
  + Original dataset rows appended (ORIG_ROW_WEIGHT = 0.35)
  + Multi-seed averaging (5 seeds × 5 folds = 25 total)
  + Logit-space bias tuning (replaces Optuna class-weight search)
    — more principled than probability-space class weight scaling
  + Hard-example diagnostics (pre and post bias)
  + magic_score=0 boundary analysis
  + W&B integration (all 4 points: artifacts, fold tracking,
    confusion matrices, config)
  + Telegram notifier + heartbeat
  + Asymmetric pseudolabelling (post-bias, downweighted)
  + Per-class OOF BA breakdown

  LABEL MAPPING
  ─────────────
  Low=0  Medium=1  High=2  (matches XGB/CatBoost convention)

SAVE CONVENTION
───────────────
  oof_lgbm_v1.npy         (n_competition, 3)  Low=0 Med=1 High=2
  pred_lgbm_v1.npy        (n_test, 3)
  oof_lgbm_v1_biased.npy  (alias)
  pred_lgbm_v1_biased.npy (alias)
"""

# ============================================================
# SETUP (run once if needed)
# !pip install lightgbm wandb -q
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
import gc
import os
import json
import time
import random
import hashlib
import warnings
import threading
import traceback
import urllib.request
from contextlib import contextmanager

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from google.colab import userdata
from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import balanced_accuracy_score
from scipy.special import logit

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# ============================================================
# SECTION 0 — CONFIGURATION
# ============================================================

# ── Paths ─────────────────────────────────────────────────────
COMP_PATH  = "/content/drive/MyDrive/irrigation_need_v15/"
OOF_DIR    = "/content/drive/MyDrive/irrigation_need_v15/"
WORK_DIR   = "/content/drive/MyDrive/irrigation_need_v15/"
SUB_DIR    = "/content/"

TRAIN_PATH    = f"{COMP_PATH}train.csv"
TEST_PATH     = f"{COMP_PATH}test.csv"
ORIGINAL_PATH = f"{COMP_PATH}irrigation_prediction.csv"
HARD_IDX_PATH = f"{COMP_PATH}hard_example_indices.npy"

# ── W&B ──────────────────────────────────────────────────────
WANDB_PROJECT = "ps-s6e4-irrigation"
WANDB_ENTITY  = "wblackstone-twilight-signals"
WANDB_ENABLED = True
WB_TOKEN      = "WB_TOKEN"   # Colab secret key name

# ── Telegram ─────────────────────────────────────────────────
TELEGRAM_BOT_TOKEN = "8755783601:AAGuCUzM6CdgjA825tep5f5zj0FNd7iSkp4"
TELEGRAM_CHAT_ID   = "5422067007"
TELEGRAM_ENABLED   = True

# ── Run identity ─────────────────────────────────────────────
TAG      = "lgbm_v1"
RUN_NAME = "LGBM v1"

# ── CV ───────────────────────────────────────────────────────
SEEDS     = [42, 123, 2024, 7, 314]
N_FOLDS   = 5
N_CLASSES = 3

# ── Target ───────────────────────────────────────────────────
TARGET             = "Irrigation_Need"
TARGET_MAPPING     = {"Low": 0, "Medium": 1, "High": 2}
INV_TARGET_MAPPING = {0: "Low", 1: "Medium", 2: "High"}
CLASS_NAMES        = ["Low", "Medium", "High"]

# ── Feature settings ─────────────────────────────────────────
ORIG_ROW_WEIGHT  = 0.35   # downweight original dataset rows
FREQ_MIN_COUNT   = 5      # minimum occurrences for freq encoding
DIGIT_RANGE      = range(-4, 4)   # k values for digit extraction

# ── Physical thresholds (for magic_score diagnostic) ─────────
SOIL_THRESH = 25
RAIN_THRESH = 300
TEMP_THRESH = 30
WIND_THRESH = 10

# ── Pseudolabelling ───────────────────────────────────────────
PSEUDO_THRESH_BY_CLASS = {
    0: 0.97,   # Low
    1: 0.80,   # Medium
    2: 0.97,   # High
}
PSEUDO_WEIGHT = 0.7

# ── LightGBM params ───────────────────────────────────────────
LGB_PARAMS = {
    "n_estimators"      : 6000,
    "boosting_type"     : "gbdt",
    "max_depth"         : 4,
    "num_leaves"        : 32,
    "learning_rate"     : 0.05,
    "feature_fraction"  : 0.6,
    "bagging_fraction"  : 0.7,
    "bagging_freq"      : 1,
    "lambda_l1"         : 10,
    "lambda_l2"         : 10,
    "min_child_samples" : 12,
    "random_state"      : 42,
    "n_jobs"            : -1,
    "max_bin"           : 15000,
    "verbosity"         : -1,
    "subsample"         : 0.5,
    "subsample_for_bin" : 100000,
    "subsample_freq"    : 1,
    "device"            : "gpu",
    "objective"         : "multiclass",
    "num_class"         : N_CLASSES,
    "metric"            : "multi_logloss",
}

print(f"{'='*60}")
print(f"  {RUN_NAME}")
print(f"{'='*60}")
print(f"  Seeds        : {SEEDS}")
print(f"  N_FOLDS      : {N_FOLDS}")
print(f"  Total CV     : {len(SEEDS) * N_FOLDS} folds")
print(f"  Orig weight  : {ORIG_ROW_WEIGHT}")
print(f"  Digit range  : k={list(DIGIT_RANGE)}")
print(f"{'='*60}")

# ============================================================
# SECTION 1 — W&B HELPER
# ============================================================
class WandbLogger:
    """Thin wrapper — W&B failures never crash training."""
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.run     = None

    def init(self, config):
        if not self.enabled:
            return
        try:
            api_key = userdata.get(WB_TOKEN)
            wandb.login(key=api_key, relogin=True)
            self.run = wandb.init(
                project = WANDB_PROJECT,
                entity  = WANDB_ENTITY,
                name    = TAG,
                config  = config,
                tags    = ["lgbm", "v1", "ps-s6e4"],
            )
            print(f"  W&B run: {self.run.url}")
        except Exception as e:
            print(f"  [W&B] init failed: {e}")
            self.enabled = False

    def log(self, metrics, step=None):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log(metrics, step=step) if step is not None \
                else wandb.log(metrics)
        except Exception as e:
            print(f"  [W&B] log failed: {e}")

    def log_confusion_matrix(self, y_true, y_pred, title):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log({
                title: wandb.plot.confusion_matrix(
                    probs=None,
                    y_true=y_true.tolist(),
                    preds=y_pred.tolist(),
                    class_names=CLASS_NAMES,
                )
            })
        except Exception as e:
            print(f"  [W&B] confusion matrix failed: {e}")

    def log_artifact(self, local_path, artifact_name,
                     artifact_type, description=""):
        if not self.enabled or self.run is None:
            return
        try:
            art = wandb.Artifact(
                name=artifact_name, type=artifact_type,
                description=description,
            )
            art.add_file(local_path)
            self.run.log_artifact(art)
            print(f"  [W&B] artifact logged: {artifact_name}")
        except Exception as e:
            print(f"  [W&B] artifact failed: {e}")

    def summary(self, metrics):
        if not self.enabled or self.run is None:
            return
        try:
            for k, v in metrics.items():
                wandb.run.summary[k] = v
        except Exception as e:
            print(f"  [W&B] summary failed: {e}")

    def finish(self):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.finish()
        except Exception as e:
            print(f"  [W&B] finish failed: {e}")


wb = WandbLogger(enabled=WANDB_ENABLED)

# ============================================================
# SECTION 2 — TELEGRAM NOTIFIER
# ============================================================
class TelegramNotifier:
    def __init__(self, bot_token=TELEGRAM_BOT_TOKEN,
                 chat_id=TELEGRAM_CHAT_ID,
                 enabled=TELEGRAM_ENABLED, run_name=RUN_NAME):
        self.bot_token  = bot_token
        self.chat_id    = chat_id
        self.enabled    = enabled
        self.run_name   = run_name
        self._start     = None
        self._hb_stop   = threading.Event()
        self._hb_thread = None

    def send(self, message, silent=False):
        if not self.enabled:
            return True
        try:
            url     = f"https://api.telegram.org/bot{self.bot_token}/sendMessage"
            payload = json.dumps({
                "chat_id"              : self.chat_id,
                "text"                 : message,
                "disable_notification" : silent,
            }).encode("utf-8")
            req = urllib.request.Request(
                url, data=payload,
                headers={"Content-Type": "application/json"},
            )
            urllib.request.urlopen(req, timeout=10)
            return True
        except Exception as e:
            print(f"[TELEGRAM] {e}")
            return False

    def start_timer(self):
        self._start = time.time()
        return self

    def elapsed(self):
        if self._start is None:
            return "unknown"
        s = int(time.time() - self._start)
        h, r = divmod(s, 3600)
        m, s = divmod(r, 60)
        return f"{h}h {m}m {s}s" if h else f"{m}m {s}s"

    def heartbeat(self, interval_minutes=20):
        if not self.enabled:
            return self
        if self._hb_thread is not None:
            self._hb_stop.set()
        self._hb_stop = threading.Event()
        def _loop():
            count = 0
            while not self._hb_stop.wait(interval_minutes * 60):
                count += 1
                self.send(
                    f"[{self.run_name}] running | "
                    f"{self.elapsed()} | hb#{count}",
                    silent=True,
                )
        self._hb_thread = threading.Thread(target=_loop, daemon=True)
        self._hb_thread.start()
        return self

    def stop_heartbeat(self):
        if self._hb_stop:
            self._hb_stop.set()

    def notify_seed(self, seed_idx, n_seeds, seed, seed_ba, silent=True):
        bar = "X" * seed_idx + "." * (n_seeds - seed_idx)
        self.send(
            f"[{self.run_name}] Seed {seed_idx}/{n_seeds} [{bar}]\n"
            f"  Seed: {seed} | OOF BA: {seed_ba:.6f} | {self.elapsed()}",
            silent=silent,
        )

    def success(self, oof_score, extra=""):
        self.stop_heartbeat()
        msg = (f"[{self.run_name}] Complete\n"
               f"  OOF BA : {oof_score:.6f}\n"
               f"  Runtime: {self.elapsed()}")
        if extra:
            msg += f"\n  {extra}"
        self.send(msg)

    def failure(self, exc=None, context=""):
        self.stop_heartbeat()
        tb = traceback.format_exc() if exc else ""
        if len(tb) > 800:
            tb = "..." + tb[-800:]
        lines = [f"[{self.run_name}] FAILED",
                 f"  Elapsed: {self.elapsed()}"]
        if context:
            lines.append(f"  Context: {context}")
        if exc:
            lines.append(f"  {type(exc).__name__}: {exc}")
        if tb:
            lines.append(tb)
        self.send("\n".join(lines))

    @contextmanager
    def run_context(self, context=""):
        try:
            yield
        except Exception as e:
            self.failure(exc=e, context=context)
            raise


notifier = TelegramNotifier()

# ============================================================
# SECTION 3 — REPRODUCIBILITY
# ============================================================
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_everything()

# ============================================================
# SECTION 4 — CUSTOM EVAL METRIC
# ============================================================
def lgb_bal_acc(y_true, y_pred):
    """Custom LightGBM eval metric — balanced accuracy."""
    preds = y_pred.reshape(N_CLASSES, -1).T
    score = balanced_accuracy_score(y_true.astype(int), preds.argmax(axis=1))
    return "bal_acc", score, True

# ============================================================
# SECTION 5 — FEATURE ENGINEERING
# ============================================================
def engineer_features(df: pd.DataFrame, max_vals: pd.Series) -> pd.DataFrame:
    """
    Extract digit features from all numerical columns.
    Uses k=-4..3 range (8 digits per column = 88 features total).
    Rounds remaining numerical features based on magnitude.
    """
    df = df.copy()
    for c in NUMS:
        for k in DIGIT_RANGE:
            df[f"{c}_digit{k}"] = (
                (df[c] // (10**k)) % 10
            ).astype("int8")
        # Round numerical values based on magnitude (same as baseline)
        if max_vals[c] < 10:
            df[c] = df[c].round(3)
        elif max_vals[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)
    return df

# ============================================================
# SECTION 6 — BIAS TUNING
# ============================================================
def tune_logit_bias(oof_probs, y_true):
    def get_preds(probs, bias):
        adj = logit(np.clip(probs, 1e-15, 1-1e-15)) + bias
        return np.argmax(adj, axis=1)

    best_bias   = np.zeros(3)
    best_score  = balanced_accuracy_score(y_true, oof_probs.argmax(1))
    raw_score   = best_score
    opt_history = [best_score]

    for step in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]:
        improved = True
        while improved:
            improved = False
            for ci in range(3):
                for d in [1, -1]:
                    trial      = best_bias.copy()
                    trial[ci] += d * step
                    s = balanced_accuracy_score(
                        y_true, get_preds(oof_probs, trial)
                    )
                    if s > best_score + 1e-9:
                        best_score, best_bias, improved = s, trial, True
                        opt_history.append(best_score)

    print(f"  Bias raw  : {raw_score:.6f}")
    print(f"  Bias tuned: {best_score:.6f} "
          f"(+{best_score - raw_score:.6f})")
    print(f"  Biases    : Low={best_bias[0]:.4f} "
          f"Med={best_bias[1]:.4f} High={best_bias[2]:.4f}")
    return best_bias, best_score, opt_history


def apply_bias(probs, bias):
    log_p = logit(np.clip(probs, 1e-15, 1-1e-15)) + bias
    exp_p = np.exp(log_p)
    return (exp_p / exp_p.sum(axis=1, keepdims=True)).astype(np.float32)

# ============================================================
# SECTION 7 — MAIN PIPELINE
# ============================================================
if __name__ == "__main__":
    notifier.start_timer().heartbeat(interval_minutes=20)

    # ── W&B init ─────────────────────────────────────────────
    wb_config = {
        "seeds"               : SEEDS,
        "n_folds"             : N_FOLDS,
        "total_cv_folds"      : len(SEEDS) * N_FOLDS,
        "orig_row_weight"     : ORIG_ROW_WEIGHT,
        "freq_min_count"      : FREQ_MIN_COUNT,
        "digit_range"         : f"k={list(DIGIT_RANGE)}",
        "pseudo_thresh_low"   : PSEUDO_THRESH_BY_CLASS[0],
        "pseudo_thresh_medium": PSEUDO_THRESH_BY_CLASS[1],
        "pseudo_thresh_high"  : PSEUDO_THRESH_BY_CLASS[2],
        "pseudo_weight"       : PSEUDO_WEIGHT,
        "tag"                 : TAG,
        "version"             : "v1",
        **{f"lgb_{k}": v for k, v in LGB_PARAMS.items()},
    }
    wb.init(config=wb_config)

    notifier.send(
        f"[{RUN_NAME}] Starting\n"
        f"  Seeds: {SEEDS}\n"
        f"  {len(SEEDS)} × {N_FOLDS}-fold = "
        f"{len(SEEDS)*N_FOLDS} folds",
        silent=True,
    )

    try:
        # ── 1. Load data ──────────────────────────────────────
        print(f"\n[1] Loading data...")
        train_raw = pd.read_csv(TRAIN_PATH)
        test_raw  = pd.read_csv(TEST_PATH)
        orig_raw  = pd.read_csv(ORIGINAL_PATH)

        if "Irrigation_Requirement" in orig_raw.columns:
            orig_raw = orig_raw.rename(
                columns={"Irrigation_Requirement": TARGET}
            )

        # Fixed label mapping — matches XGB/CatBoost convention
        train_raw[TARGET] = train_raw[TARGET].map(TARGET_MAPPING)
        orig_raw[TARGET]  = orig_raw[TARGET].map(TARGET_MAPPING)

        # Assign new IDs to original rows
        max_id = int(train_raw["id"].max())
        orig_raw["id"] = range(max_id + 1, max_id + 1 + len(orig_raw))

        n_competition = len(train_raw)
        test_ids      = test_raw["id"].copy()

        # Drop id column before feature engineering
        train_raw = train_raw.drop("id", axis=1)
        test_raw_fe = test_raw.drop("id", axis=1)
        orig_raw  = orig_raw.drop("id", axis=1)

        CATS = [c for c in test_raw_fe.columns
                if test_raw_fe[c].dtype == object]
        NUMS = [c for c in test_raw_fe.columns
                if c not in CATS]

        print(f"  Competition rows : {n_competition:,}")
        print(f"  Original rows    : {len(orig_raw):,}")
        print(f"  Test rows        : {len(test_raw_fe):,}")
        print(f"  CATS             : {len(CATS)}")
        print(f"  NUMS             : {len(NUMS)}")

        # ── 2. Load hard example indices ─────────────────────
        hard_mask = None
        if os.path.exists(HARD_IDX_PATH):
            hard_idx  = np.load(HARD_IDX_PATH)
            hard_mask = np.zeros(n_competition, dtype=bool)
            hard_mask[hard_idx] = True
            print(f"  Hard examples    : {hard_mask.sum():,} rows ✅")
        else:
            print(f"  Hard examples    : not found — skipping")

        # ── 3. Feature engineering ────────────────────────────
        print(f"\n[2] Feature engineering (digit k={list(DIGIT_RANGE)})...")

        # Compute max values from competition train only (same as baseline)
        max_vals = train_raw[NUMS].max()

        train_eng = engineer_features(train_raw, max_vals)
        test_eng  = engineer_features(test_raw_fe, max_vals)
        orig_eng  = engineer_features(orig_raw, max_vals)

        # Drop zero-variance digit columns (global check across train)
        digit_cols = [c for c in test_eng.columns if "digit" in c]
        zero_var_global = [
            c for c in digit_cols
            if train_eng[c].nunique() == 1
        ]
        if zero_var_global:
            print(f"  Dropping {len(zero_var_global)} zero-variance "
                  f"digit cols: {zero_var_global}")
            train_eng = train_eng.drop(columns=zero_var_global)
            test_eng  = test_eng.drop(columns=zero_var_global)
            orig_eng  = orig_eng.drop(columns=zero_var_global)
        else:
            print(f"  ✅ No zero-variance digit columns found")

        # Hard-row zero-variance check on digit features
        if hard_mask is not None:
            digit_cols_remaining = [
                c for c in train_eng.columns if "digit" in c
            ]
            hard_zero_var = [
                c for c in digit_cols_remaining
                if train_eng[hard_mask][c].nunique() == 1
            ]
            if hard_zero_var:
                print(f"  ⚠  Zero-variance on hard rows: {hard_zero_var}")
            else:
                print(f"  ✅ Zero-variance check on hard rows passed")

        # Recompute digit cols after dropping
        digit_cols = [c for c in test_eng.columns if "digit" in c]

        # ── 4. Frequency-rank categorical encoding ────────────
        print(f"\n[3] Frequency-rank encoding "
              f"(CATS + digit features, min={FREQ_MIN_COUNT})...")

        CATEGORY = CATS + digit_cols
        freq_mappings = {}

        for c in CATEGORY:
            freq   = train_eng[c].value_counts()
            mapping = {
                val: idx
                for idx, (val, count) in enumerate(
                    freq[freq >= FREQ_MIN_COUNT].items()
                )
            }
            default = len(mapping)
            freq_mappings[c] = (mapping, default)

            train_eng[c] = train_eng[c].map(
                lambda x, m=mapping, d=default: m.get(x, d)
            )
            test_eng[c]  = test_eng[c].map(
                lambda x, m=mapping, d=default: m.get(x, d)
            )
            orig_eng[c]  = orig_eng[c].map(
                lambda x, m=mapping, d=default: m.get(x, d)
            )

        FEATURES = CATEGORY + NUMS
        print(f"  Category features: {len(CATEGORY)}")
        print(f"  Total features   : {len(FEATURES)}")
        wb.log({"n_features": len(FEATURES),
                "n_digit_features": len(digit_cols),
                "n_category_features": len(CATEGORY)})

        # ── 5. Labels and sample weights ─────────────────────
        y_train   = train_eng[TARGET].values.astype(int)
        y_orig    = orig_eng[TARGET].values.astype(int)

        X_train   = train_eng.drop(columns=[TARGET])
        X_test    = test_eng.copy()
        X_orig    = orig_eng.drop(columns=[TARGET])

        # Balanced sample weights for competition rows
        sw_train  = compute_sample_weight("balanced", y_train)
        # Original rows get downweighted
        sw_orig   = compute_sample_weight("balanced", y_orig) * ORIG_ROW_WEIGHT

        print(f"\n  X_train : {X_train.shape}")
        print(f"  X_orig  : {X_orig.shape}")
        print(f"  X_test  : {X_test.shape}")
        print(f"  Class dist: "
              f"{dict(zip(*np.unique(y_train, return_counts=True)))}")

        wb.log({
            "n_competition": n_competition,
            "n_orig_rows"  : len(orig_raw),
        })

        notifier.send(
            f"[{RUN_NAME}] Data ready\n"
            f"  Features: {len(FEATURES)} | "
            f"Orig: {len(orig_raw):,}\n"
            f"  Elapsed: {notifier.elapsed()}",
            silent=True,
        )

        # ── 6. Multi-seed CV ──────────────────────────────────
        print(f"\n[4] Multi-seed CV "
              f"({len(SEEDS)} seeds × {N_FOLDS} folds)...")

        oof_accum       = np.zeros(
            (n_competition, N_CLASSES), dtype=np.float64
        )
        test_accum      = np.zeros(
            (len(X_test), N_CLASSES), dtype=np.float64
        )
        seed_oof_scores = []
        all_best_iters  = []
        total_start     = time.time()
        global_fold_num = 0   # monotonic W&B step counter

        for seed_idx, seed in enumerate(SEEDS):
            seed_start = time.time()
            print(f"\n{'─'*60}")
            print(f"  SEED {seed}  ({seed_idx+1}/{len(SEEDS)})")
            print(f"{'─'*60}")

            seed_everything(seed)
            kf = KFold(
                n_splits=N_FOLDS, shuffle=True, random_state=seed
            )

            oof_seed  = np.zeros(
                (n_competition, N_CLASSES), dtype=np.float64
            )
            test_seed = np.zeros(
                (len(X_test), N_CLASSES), dtype=np.float64
            )
            seed_iters = []

            for fold, (tr_idx, val_idx) in enumerate(
                kf.split(X_train)
            ):
                fold_start = time.time()
                print(f"\n  Fold {fold+1}/{N_FOLDS} | Seed {seed}")

                with notifier.run_context(
                    f"Seed {seed} Fold {fold+1}"
                ):
                    X_tr  = X_train.iloc[tr_idx].copy()
                    y_tr  = y_train[tr_idx]
                    X_va  = X_train.iloc[val_idx].copy()
                    y_va  = y_train[val_idx]
                    sw_tr = sw_train[tr_idx]

                    # ── TargetEncoder on all FEATURES ─────────
                    te = TargetEncoder(
                        target_type="multiclass",
                        smooth="auto",
                        cv=5,
                        random_state=seed,
                    )
                    X_tr_enc = pd.DataFrame(
                        te.fit_transform(X_tr[FEATURES], y_tr),
                        index=X_tr.index,
                    )
                    X_va_enc = pd.DataFrame(
                        te.transform(X_va[FEATURES]),
                        index=X_va.index,
                    )
                    X_te_enc = pd.DataFrame(
                        te.transform(X_test[FEATURES]),
                    )
                    X_og_enc = pd.DataFrame(
                        te.transform(X_orig[FEATURES]),
                    )

                    # Concat TE features alongside raw NUMS
                    X_tr_full = pd.concat(
                        [X_tr, X_tr_enc], axis=1
                    ).drop(columns=CATS)
                    X_va_full = pd.concat(
                        [X_va, X_va_enc], axis=1
                    ).drop(columns=CATS)
                    X_te_full = pd.concat(
                        [X_test.reset_index(drop=True), X_te_enc],
                        axis=1,
                    ).drop(columns=CATS)
                    X_og_full = pd.concat(
                        [X_orig.reset_index(drop=True), X_og_enc],
                        axis=1,
                    ).drop(columns=CATS)

                    # Append original rows to training fold
                    X_tr_merged = pd.concat(
                        [X_tr_full, X_og_full],
                        axis=0, ignore_index=True,
                    )
                    y_tr_merged = np.concatenate([y_tr, y_orig])
                    sw_merged   = np.concatenate([sw_tr, sw_orig])

                    print(f"    train: {len(X_tr_merged):,} "
                          f"(comp: {len(y_tr):,} + "
                          f"orig: {len(y_orig):,}) | "
                          f"val: {len(y_va):,}")

                    model = LGBMClassifier(
                        **LGB_PARAMS, random_state=seed
                    )
                    model.fit(
                        X_tr_merged, y_tr_merged,
                        sample_weight=sw_merged,
                        eval_set=[(X_va_full, y_va)],
                        eval_metric=lgb_bal_acc,
                        callbacks=[
                            log_evaluation(200),
                            early_stopping(250, verbose=False),
                        ],
                    )

                    best_iter = model.best_iteration_
                    seed_iters.append(best_iter)
                    all_best_iters.append(best_iter)

                    val_proba  = model.predict_proba(X_va_full)
                    test_proba = model.predict_proba(X_te_full)

                    oof_seed[val_idx] += val_proba
                    test_seed         += test_proba

                    fold_ba  = balanced_accuracy_score(
                        y_va, val_proba.argmax(axis=1)
                    )
                    elapsed  = int(time.time() - fold_start)
                    print(f"    BA={fold_ba:.5f} | "
                          f"iter={best_iter} | "
                          f"{elapsed//60}m {elapsed%60}s")

                    # W&B fold metrics — monotonic step
                    global_fold_num += 1
                    wb.log({
                        "fold_ba"       : fold_ba,
                        "fold_best_iter": best_iter,
                        "fold_elapsed_s": elapsed,
                        "seed"          : seed,
                        "fold"          : fold + 1,
                        "seed_idx"      : seed_idx + 1,
                        "global_fold"   : global_fold_num,
                    }, step=global_fold_num)

                    del model
                    del X_tr_full, X_va_full, X_te_full, X_og_full
                    del X_tr_enc, X_va_enc, X_te_enc, X_og_enc
                    del X_tr_merged
                    gc.collect()

            test_seed /= N_FOLDS
            seed_ba    = balanced_accuracy_score(
                y_train, oof_seed.argmax(1)
            )
            seed_oof_scores.append(seed_ba)

            seed_elapsed = int(time.time() - seed_start)
            print(f"\n  Seed {seed} OOF BA: {seed_ba:.6f} | "
                  f"Avg iter: {int(np.mean(seed_iters))} | "
                  f"{seed_elapsed//60}m {seed_elapsed%60}s")

            # Seed-level metrics — no step (summary panel)
            wb.log({
                f"seed_{seed}_oof_ba": seed_ba,
                "seed_oof_ba"        : seed_ba,
                "seed_avg_iter"      : int(np.mean(seed_iters)),
                "seed_elapsed_s"     : seed_elapsed,
            })

            notifier.notify_seed(
                seed_idx+1, len(SEEDS), seed, seed_ba
            )

            oof_accum  += oof_seed
            test_accum += test_seed

        oof_accum  /= len(SEEDS)
        test_accum /= len(SEEDS)
        avg_iter    = int(np.mean(all_best_iters))

        total_elapsed = int(time.time() - total_start)
        print(f"\n{'='*60}")
        print(f"Multi-seed CV complete | "
              f"{total_elapsed//3600}h "
              f"{(total_elapsed%3600)//60}m {total_elapsed%60}s")
        print(f"Per-seed OOF BAs : "
              f"{[round(s, 5) for s in seed_oof_scores]}")
        print(f"Mean seed BA     : {np.mean(seed_oof_scores):.6f} "
              f"± {np.std(seed_oof_scores):.6f}")

        # ── 7. Bias tuning ────────────────────────────────────
        print(f"\n[5] Bias tuning on averaged OOF...")

        raw_ba = balanced_accuracy_score(
            y_train, oof_accum.argmax(1)
        )
        print(f"  Averaged OOF raw BA: {raw_ba:.6f}")

        best_bias, tuned_ba, opt_history = tune_logit_bias(
            oof_accum.astype(np.float32), y_train
        )

        oof_calibrated  = apply_bias(
            oof_accum.astype(np.float32), best_bias
        )
        test_calibrated = apply_bias(
            test_accum.astype(np.float32), best_bias
        )

        # Per-class BA post-bias
        per_class_ba = {}
        print(f"\n  Per-class OOF BA (post-bias):")
        for cls in range(3):
            mask = y_train == cls
            ba   = (oof_calibrated[mask].argmax(axis=1) == cls).mean()
            per_class_ba[f"oof_ba_{CLASS_NAMES[cls].lower()}"] = float(ba)
            print(f"    {CLASS_NAMES[cls]:<8}: {ba:.5f}")

        wb.log({
            "oof_ba_raw"      : raw_ba,
            "oof_ba_biased"   : tuned_ba,
            "bias_correction" : tuned_ba - raw_ba,
            "avg_best_iter"   : avg_iter,
            "mean_seed_ba"    : float(np.mean(seed_oof_scores)),
            "std_seed_ba"     : float(np.std(seed_oof_scores)),
            "bias_low"        : float(best_bias[0]),
            "bias_medium"     : float(best_bias[1]),
            "bias_high"       : float(best_bias[2]),
            **per_class_ba,
        })

        # Full OOF confusion matrix
        wb.log_confusion_matrix(
            y_true=y_train,
            y_pred=oof_calibrated.argmax(axis=1),
            title="oof_confusion_matrix",
        )

        # ── 8. Hard-example analysis ──────────────────────────
        if hard_mask is not None:
            hard_oof_raw  = oof_accum[hard_mask].astype(np.float32)
            hard_oof_bias = oof_calibrated[hard_mask]
            hard_true     = y_train[hard_mask]

            hard_ba_pre  = balanced_accuracy_score(
                hard_true, hard_oof_raw.argmax(axis=1)
            )
            hard_ba_post = balanced_accuracy_score(
                hard_true, hard_oof_bias.argmax(axis=1)
            )
            print(f"\n  Hard-example OOF BA pre-bias : {hard_ba_pre:.5f}")
            print(f"  Hard-example OOF BA post-bias: {hard_ba_post:.5f}")
            print(f"\n  Hard-row per-class:")
            for cls in range(3):
                m = hard_true == cls
                if m.sum() == 0:
                    continue
                pre  = (hard_oof_raw[m].argmax(1) == cls).mean()
                post = (hard_oof_bias[m].argmax(1) == cls).mean()
                print(f"    {CLASS_NAMES[cls]:<8}: "
                      f"{pre:.5f} → {post:.5f}  "
                      f"({post-pre:+.5f})")

            wb.log({
                "hard_oof_ba_pre_bias" : hard_ba_pre,
                "hard_oof_ba_post_bias": hard_ba_post,
            })
            wb.log_confusion_matrix(
                y_true=hard_true,
                y_pred=hard_oof_bias.argmax(axis=1),
                title="hard_example_confusion_matrix",
            )

        # ── 9. magic_score=0 boundary analysis ───────────────
        print(f"\n[6] magic_score=0 boundary analysis...")
        train_ms_df = pd.read_csv(TRAIN_PATH)
        s  = (train_ms_df["Soil_Moisture"]     < SOIL_THRESH).astype(int)
        r  = (train_ms_df["Rainfall_mm"]       < RAIN_THRESH).astype(int)
        t  = (train_ms_df["Temperature_C"]     > TEMP_THRESH).astype(int)
        w  = (train_ms_df["Wind_Speed_kmh"]    > WIND_THRESH).astype(int)
        h  = (train_ms_df["Crop_Growth_Stage"] == "Harvest").astype(int)
        sw = (train_ms_df["Crop_Growth_Stage"] == "Sowing").astype(int)
        m  = (train_ms_df["Mulching_Used"]     == "Yes").astype(int)
        ms = (2*s + 2*r + t + w - 2*h - 2*sw - m).values

        ms0_mask     = ms == 0
        y_ms0        = y_train[ms0_mask]
        p_ms0_raw    = oof_accum[ms0_mask].astype(np.float32)
        p_ms0_tuned  = oof_calibrated[ms0_mask]

        print(f"  magic_score=0 rows : {ms0_mask.sum():,}")
        med_idx          = 1   # Medium = 1 in our mapping
        med_recall_raw   = (p_ms0_raw[y_ms0==med_idx].argmax(1) == med_idx).mean()
        med_recall_tuned = (p_ms0_tuned[y_ms0==med_idx].argmax(1) == med_idx).mean()
        print(f"  Medium recall (raw)  : {med_recall_raw:.5f}")
        print(f"  Medium recall (tuned): {med_recall_tuned:.5f}")
        wb.log({
            "ms0_medium_recall_raw"  : float(med_recall_raw),
            "ms0_medium_recall_tuned": float(med_recall_tuned),
        })
        del train_ms_df
        gc.collect()

        # ── 10. Diagnostic plots ──────────────────────────────
        print(f"\n[7] Generating diagnostic plots...")

        fig1, ax1 = plt.subplots(figsize=(9, 4))
        bars = ax1.bar(
            [str(s) for s in SEEDS], seed_oof_scores,
            color="teal", alpha=0.8, edgecolor="black",
        )
        ax1.axhline(
            np.mean(seed_oof_scores), color="red",
            linestyle="--",
            label=f"Mean: {np.mean(seed_oof_scores):.5f}",
        )
        for bar, val in zip(bars, seed_oof_scores):
            ax1.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.0001,
                f"{val:.5f}", ha="center", va="bottom", fontsize=9,
            )
        ax1.set_title(
            f"Per-Seed OOF BA ({RUN_NAME})", fontweight="bold"
        )
        ax1.set_xlabel("Seed"); ax1.set_ylabel("Balanced Accuracy")
        ax1.legend(); plt.tight_layout()
        wb.log({"chart_seed_oof_ba": wandb.Image(fig1)})
        plt.show(); plt.close(fig1)

        fig2, ax2 = plt.subplots(figsize=(10, 4))
        ax2.plot(
            range(len(opt_history)), opt_history,
            color="teal", marker="o", markersize=4, linewidth=1.5,
        )
        ax2.axhline(
            raw_ba, color="gray", linestyle="--",
            label="Raw averaged OOF BA",
        )
        ax2.set_title(
            f"Bias Tuning: {raw_ba:.5f} → {tuned_ba:.5f} "
            f"(+{tuned_ba - raw_ba:.5f})", fontweight="bold",
        )
        ax2.set_xlabel("Step"); ax2.set_ylabel("Balanced Accuracy")
        ax2.legend(); plt.tight_layout()
        wb.log({"chart_bias_tuning": wandb.Image(fig2)})
        plt.show(); plt.close(fig2)

        fig3, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.kdeplot(
            oof_calibrated.max(1), ax=axes[0], label="OOF",
            fill=True, color="teal", alpha=0.5,
        )
        sns.kdeplot(
            test_calibrated.max(1), ax=axes[0], label="Test",
            fill=True, color="orange", alpha=0.3,
        )
        axes[0].set_title("Confidence Distribution")
        axes[0].legend()
        oof_dist  = pd.Series(
            [INV_TARGET_MAPPING[p]
             for p in oof_calibrated.argmax(1)]
        ).value_counts(normalize=True).sort_index()
        test_dist = pd.Series(
            [INV_TARGET_MAPPING[p]
             for p in test_calibrated.argmax(1)]
        ).value_counts(normalize=True).sort_index()
        x = np.arange(3)
        axes[1].bar(x-0.2, oof_dist.values,  0.4,
                    label="OOF",  color="teal",   alpha=0.7)
        axes[1].bar(x+0.2, test_dist.values, 0.4,
                    label="Test", color="orange", alpha=0.7)
        axes[1].set_title("Class Distribution")
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(oof_dist.index)
        axes[1].legend()
        plt.tight_layout()
        wb.log({"chart_confidence_dist": wandb.Image(fig3)})
        plt.show(); plt.close(fig3)

        # ── 11. Pseudolabelling ───────────────────────────────
        print(f"\n[8] Pseudolabeling (asymmetric thresholds)...")
        print(f"  Thresholds: Low={PSEUDO_THRESH_BY_CLASS[0]} | "
              f"Medium={PSEUDO_THRESH_BY_CLASS[1]} | "
              f"High={PSEUDO_THRESH_BY_CLASS[2]}")

        pseudo_labels_all = test_calibrated.argmax(axis=1)
        pseudo_conf_all   = test_calibrated.max(axis=1)
        pseudo_mask_arr   = np.array([
            pseudo_conf_all[i] >=
            PSEUDO_THRESH_BY_CLASS[pseudo_labels_all[i]]
            for i in range(len(test_calibrated))
        ])
        pseudo_labels     = pseudo_labels_all[pseudo_mask_arr]

        print(f"  High-confidence test samples: "
              f"{pseudo_mask_arr.sum():,}")
        pseudo_counts = {}
        for cls in range(3):
            n = (pseudo_labels == cls).sum()
            pseudo_counts[
                f"pseudo_n_{CLASS_NAMES[cls].lower()}"
            ] = int(n)
            print(f"    {INV_TARGET_MAPPING[cls]}: {n:,}")

        wb.log({
            "pseudo_n_total": int(pseudo_mask_arr.sum()),
            **pseudo_counts,
        })

        if pseudo_mask_arr.sum() > 500:
            # Re-encode test pseudo rows with same TE
            # Use seed=SEEDS[0] for reproducibility
            seed_everything(SEEDS[0])
            te_pseudo = TargetEncoder(
                target_type="multiclass",
                smooth="auto", cv=5,
                random_state=SEEDS[0],
            )
            X_full_enc = pd.DataFrame(
                te_pseudo.fit_transform(
                    X_train[FEATURES], y_train
                )
            )
            X_test_enc_ps = pd.DataFrame(
                te_pseudo.transform(X_test[FEATURES])
            )
            X_orig_enc_ps = pd.DataFrame(
                te_pseudo.transform(X_orig[FEATURES])
            )

            X_full_ps = pd.concat(
                [X_train, X_full_enc], axis=1
            ).drop(columns=CATS).reset_index(drop=True)
            X_test_ps = pd.concat(
                [X_test.reset_index(drop=True), X_test_enc_ps],
                axis=1,
            ).drop(columns=CATS)
            X_orig_ps = pd.concat(
                [X_orig.reset_index(drop=True), X_orig_enc_ps],
                axis=1,
            ).drop(columns=CATS)

            X_aug = pd.concat(
                [X_full_ps,
                 X_orig_ps,
                 X_test_ps[pseudo_mask_arr]],
                axis=0, ignore_index=True,
            )
            y_aug = np.concatenate([
                y_train, y_orig, pseudo_labels
            ])
            sw_comp  = compute_sample_weight("balanced", y_train)
            sw_ps    = np.full(pseudo_mask_arr.sum(), PSEUDO_WEIGHT)
            sw_aug   = np.concatenate([sw_comp, sw_orig, sw_ps])

            print(f"  Retraining on {len(X_aug):,} rows "
                  f"({pseudo_mask_arr.sum():,} pseudo) "
                  f"for {avg_iter} rounds...")

            model_ps = LGBMClassifier(
                **{**LGB_PARAMS,
                   "n_estimators": avg_iter,
                   "random_state": SEEDS[0]},
            )
            model_ps.fit(
                X_aug, y_aug,
                sample_weight=sw_aug,
                callbacks=[log_evaluation(100)],
            )
            final_test_probs = model_ps.predict_proba(
                X_test_ps
            ).astype(np.float32)
            final_test_probs = apply_bias(
                final_test_probs, best_bias
            )
            print("  Pseudolabeling complete.")
            del model_ps, X_aug
            gc.collect()
        else:
            final_test_probs = test_calibrated
            print("  Skipped — insufficient high-confidence samples.")

        # ── 12. Save + W&B Artifacts ──────────────────────────
        print(f"\n[9] Saving...")

        oof_path  = f"{OOF_DIR}oof_{TAG}.npy"
        pred_path = f"{OOF_DIR}pred_{TAG}.npy"

        np.save(oof_path,
                oof_calibrated.astype(np.float32))
        np.save(pred_path,
                final_test_probs.astype(np.float32))
        np.save(f"{OOF_DIR}oof_{TAG}_biased.npy",
                oof_calibrated.astype(np.float32))
        np.save(f"{OOF_DIR}pred_{TAG}_biased.npy",
                final_test_probs.astype(np.float32))
        # Also save raw (pre-bias) for ensemble diagnostics
        np.save(f"{OOF_DIR}oof_{TAG}_raw.npy",
                oof_accum.astype(np.float32))

        assert oof_calibrated.shape   == (n_competition, N_CLASSES), \
            f"OOF shape: {oof_calibrated.shape}"
        assert final_test_probs.shape == (len(test_ids), N_CLASSES), \
            f"Pred shape: {final_test_probs.shape}"
        print("  Shape assertions passed ✅")
        print(f"  oof_{TAG}.npy   {oof_calibrated.shape}")
        print(f"  pred_{TAG}.npy  {final_test_probs.shape}")

        wb.log_artifact(
            local_path    = oof_path,
            artifact_name = f"oof_{TAG}",
            artifact_type = "model_output",
            description   = f"LGBM OOF | biased BA={tuned_ba:.6f}",
        )
        wb.log_artifact(
            local_path    = pred_path,
            artifact_name = f"pred_{TAG}",
            artifact_type = "model_output",
            description   = f"LGBM pred | biased BA={tuned_ba:.6f}",
        )

        # ── 13. Submission ────────────────────────────────────
        sub = pd.DataFrame({
            "id"  : test_ids,
            TARGET: [INV_TARGET_MAPPING[p]
                     for p in final_test_probs.argmax(axis=1)],
        })
        sub_path = f"{SUB_DIR}submission_{TAG}.csv"
        sub.to_csv(sub_path, index=False)
        print(f"\n  Submission: {sub_path}")
        print(sub[TARGET].value_counts().to_string())

        wb.log({
            f"submission_n_{k.lower()}": int(v)
            for k, v in sub[TARGET].value_counts().items()
        })

        # ── 14. W&B summary + final print ─────────────────────
        wb.summary({
            "oof_ba_raw"     : raw_ba,
            "oof_ba_biased"  : tuned_ba,
            "bias_correction": tuned_ba - raw_ba,
            "mean_seed_ba"   : float(np.mean(seed_oof_scores)),
            "avg_best_iter"  : avg_iter,
            "tag"            : TAG,
        })

        print(f"\n{'='*60}")
        print(f"{RUN_NAME} SUMMARY")
        print(f"{'='*60}")
        print(f"  Seeds          : {SEEDS}")
        print(f"  Per-seed BAs   : "
              f"{[round(s, 5) for s in seed_oof_scores]}")
        print(f"  Mean seed BA   : {np.mean(seed_oof_scores):.6f} "
              f"± {np.std(seed_oof_scores):.6f}")
        print(f"  Averaged raw BA: {raw_ba:.6f}")
        print(f"  Biased BA      : {tuned_ba:.6f} "
              f"(+{tuned_ba - raw_ba:.6f})")
        print(f"  Biases         : {np.round(best_bias, 4)}")
        print(f"  Avg best iter  : {avg_iter}")
        print(f"  Features       : {len(FEATURES)}")
        print(f"  Digit range    : k={list(DIGIT_RANGE)}")
        print(f"  Runtime        : {notifier.elapsed()}")
        print(f"{'='*60}")

        notifier.success(
            oof_score=tuned_ba,
            extra=(
                f"raw={raw_ba:.5f} | "
                f"avg_iter={avg_iter} | "
                f"features={len(FEATURES)}"
            ),
        )

    except Exception as e:
        notifier.failure(exc=e, context=f"Main {RUN_NAME}")
        raise

    finally:
        wb.finish()

  LGBM v1
  Seeds        : [42, 123, 2024, 7, 314]
  N_FOLDS      : 5
  Total CV     : 25 folds
  Orig weight  : 0.35
  Digit range  : k=[-4, -3, -2, -1, 0, 1, 2, 3]


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: wblackstone (wblackstone-twilight-signals) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  W&B run: https://wandb.ai/wblackstone-twilight-signals/ps-s6e4-irrigation/runs/fa2555dj

[1] Loading data...
  Competition rows : 630,000
  Original rows    : 10,000
  Test rows        : 270,000
  CATS             : 8
  NUMS             : 11
  Hard examples    : 7,377 rows ✅

[2] Feature engineering (digit k=[-4, -3, -2, -1, 0, 1, 2, 3])...
  Dropping 22 zero-variance digit cols: ['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']
  ✅ Zero-variance check on hard rows passed

[3] Frequency